# Caso 1: предсказание задержек доставки (Olist)

**Цель практики:** собрать из 6 таблиц датасет, где одна строка = один заказ, и создать целевую переменную `is_delayed`.

## 1. Импорты и путь к данным

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")  # папка с CSV-файлами Olist

## 2. Загрузка 6 нужных таблиц
Столбцы с датами сразу читаем как даты (`parse_dates`), чтобы можно было вычитать одну дату из другой.

In [8]:
import shutil
from pathlib import Path

import kagglehub

# скачиваем датасет (он сохраняется в кэш kagglehub)
ruta_cache = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))

# копируем CSV в папку data рядом с блокнотом
DATA = Path("data")
DATA.mkdir(exist_ok=True)
for archivo in ruta_cache.glob("*.csv"):
    shutil.copy(archivo, DATA / archivo.name)

print(sorted(p.name for p in DATA.glob("*.csv")))

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [9]:
orders = pd.read_csv(
    DATA / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_estimated_delivery_date",
        "order_delivered_customer_date",
    ],
)
customers = pd.read_csv(DATA / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA / "olist_geolocation_dataset.csv")

for nombre, tabla in [("orders", orders), ("customers", customers), ("order_items", order_items),
                      ("products", products), ("sellers", sellers), ("geolocation", geolocation)]:
    print(f"{nombre:12s} {tabla.shape}")

orders       (99441, 8)
customers    (99441, 5)
order_items  (112650, 7)
products     (32951, 9)
sellers      (3095, 4)
geolocation  (1000163, 5)


## 3. Фильтрация заказов
Оставляем только доставленные заказы с известной реальной датой доставки: иначе нельзя посчитать задержку.

In [10]:
print("Заказов до фильтра:", len(orders))

orders = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].notna())
].copy()

print("Заказов после фильтра:", len(orders))

Заказов до фильтра: 99441
Заказов после фильтра: 96470


## 4. Целевая переменная и признаки из даты
- `dias_retraso` — на сколько дней фактическая доставка позже обещанной (отрицательное число = доставили раньше);
- `is_delayed` — 1, если `dias_retraso > 0`;
- `mes` и `dia_semana` (0 = понедельник) берём из даты покупки.

In [11]:
orders["dias_retraso"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days
orders["is_delayed"] = (orders["dias_retraso"] > 0).astype(int)

orders["mes"] = orders["order_purchase_timestamp"].dt.month
orders["dia_semana"] = orders["order_purchase_timestamp"].dt.dayofweek

orders[["order_id", "dias_retraso", "is_delayed", "mes", "dia_semana"]].head()

,order_id,dias_retraso,is_delayed,mes,dia_semana
0,e481f51cbdc54678b7cc49136f2d6af7,-8,0,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,-6,0,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,-18,0,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,-13,0,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,-10,0,2,1


## 5. Дисбаланс классов
Смотрим, какую долю составляют задержки. Это определит, какие метрики использовать (ROC-AUC, PR-AUC, F1).

In [12]:
print(orders["is_delayed"].value_counts())
print()
print(orders["is_delayed"].value_counts(normalize=True).round(4))

is_delayed
0    89936
1     6534
Name: count, dtype: int64

is_delayed
0    0.9323
1    0.0677
Name: proportion, dtype: float64


## 6. Координаты: одна строка на почтовый индекс
В `geolocation` у одного индекса много строк. Берём среднюю широту и долготу, чтобы при соединении не размножить заказы.

In [13]:
geo_agg = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
    .reset_index()
)

# две копии с разными названиями столбцов: для клиента и для продавца
geo_cli = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "lat": "lat_cli", "lng": "lng_cli",
})
geo_ven = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "lat": "lat_ven", "lng": "lng_ven",
})

print("Индексов в geolocation:", geolocation["geolocation_zip_code_prefix"].nunique())
print("Строк в geo_agg:", len(geo_agg))

Индексов в geolocation: 19015
Строк в geo_agg: 19015


## 7. Точка назначения: заказы + клиенты + координаты клиента

In [14]:
destino = (
    orders.merge(
        customers[["customer_id", "customer_zip_code_prefix", "customer_state"]],
        on="customer_id",
        how="inner",
    )
    .merge(geo_cli, on="customer_zip_code_prefix", how="left")
)

print("Строк в destino:", len(destino))
destino[["order_id", "customer_state", "lat_cli", "lng_cli"]].head()

Строк в destino: 96470


,order_id,customer_state,lat_cli,lng_cli
0,e481f51cbdc54678b7cc49136f2d6af7,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,BA,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,GO,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,RN,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,SP,-23.676370,-46.514627


## 8. Позиции заказа: товары, продавцы и координаты продавца
Объём товара = длина × высота × ширина.

In [15]:
products["volumen_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

items = (
    order_items[["order_id", "order_item_id", "product_id", "seller_id", "freight_value"]]
    .merge(products[["product_id", "product_weight_g", "volumen_cm3"]],
           on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]],
           on="seller_id", how="left")
    .merge(geo_ven, on="seller_zip_code_prefix", how="left")
)

print("Позиций заказов:", len(items))
items.head()

Позиций заказов: 112650


,order_id,order_item_id,product_id,seller_id,freight_value,product_weight_g,volumen_cm3,seller_zip_code_prefix,seller_state,lat_ven,lng_ven
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,13.29,650.0,3528.0,27277,SP,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,19.93,30000.0,60000.0,3471,SP,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,17.87,3050.0,14157.0,37564,MG,-22.262584,-46.171124
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.79,200.0,2400.0,14403,SP,-20.553624,-47.387359
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,18.14,3750.0,42000.0,87900,PR,-22.929384,-53.135873


## 9. Расстояние клиент – продавец (формула Хаверсина)
Функция работает сразу со столбцами целиком (векторизация numpy), поэтому цикл не нужен.

In [16]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Расстояние в км между двумя точками (широта и долгота в градусах)."""
    radio_tierra = 6371.0
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * radio_tierra * np.arcsin(np.sqrt(a))


# проверка функции: Сан-Паулу -> Рио-де-Жанейро, ожидаем около 360 км
print(round(haversine_km(-23.55, -46.63, -22.91, -43.17)), "км")

361 км


In [17]:
# добавляем координаты клиента к позициям и считаем расстояние для каждой позиции
items = items.merge(destino[["order_id", "lat_cli", "lng_cli"]], on="order_id", how="inner")

items["distancia_km"] = haversine_km(
    items["lat_cli"], items["lng_cli"], items["lat_ven"], items["lng_ven"]
)

items["distancia_km"].describe()

count    109653.000000
mean        596.257743
std         588.552134
min           0.000000
25%         185.229335
50%         431.909443
75%         791.637120
max        8677.911622
Name: distancia_km, dtype: float64

## 10. Агрегация: одна строка на заказ
Суммируем flete, вес и объём, расстояние усредняем. Дополнительно считаем число позиций и продавцов.

In [18]:
items_por_orden = items.groupby("order_id").agg(
    flete=("freight_value", "sum"),
    peso_g=("product_weight_g", "sum"),
    volumen_cm3=("volumen_cm3", "sum"),
    distancia_km=("distancia_km", "mean"),
    n_items=("order_item_id", "count"),
    n_vendedores=("seller_id", "nunique"),
    seller_state=("seller_state", "first"),
).reset_index()

items_por_orden.head()

,order_id,flete,peso_g,volumen_cm3,distancia_km,n_items,n_vendedores,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,13.29,650.0,3528.0,301.504681,1,1,SP
1,00018f77f2f0320c557190d7a144bdd3,19.93,30000.0,60000.0,585.563937,1,1,SP
2,000229ec398224ef6ca0657da4fc703e,17.87,3050.0,14157.0,312.343511,1,1,MG
3,00024acbcdf0a6daa1e931b038114c75,12.79,200.0,2400.0,293.168420,1,1,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,18.14,3750.0,42000.0,646.163463,1,1,PR


## 11. Итоговый DataFrame и проверки

In [19]:
df = destino.merge(items_por_orden, on="order_id", how="inner")

print("Строк:", len(df), "| уникальных order_id:", df["order_id"].nunique())
assert len(df) == df["order_id"].nunique(), "Есть дубликаты заказов: проверь соединения"

print("\nПропуски по столбцам:")
print(df.isna().sum()[lambda s: s > 0])

Строк: 96470 | уникальных order_id: 96470

Пропуски по столбцам:
order_approved_at                14
order_delivered_carrier_date      1
lat_cli                         264
lng_cli                         264
distancia_km                    476
dtype: int64


## 12. Столбцы для модели (без утечки данных)
Убираем реальную дату доставки и `dias_retraso`: в момент покупки их ещё нет.

In [20]:
columnas_modelo = [
    "is_delayed",
    "distancia_km",
    "peso_g",
    "volumen_cm3",
    "customer_state",
    "seller_state",
    "mes",
    "dia_semana",
    "flete",
]
df_modelo = df[columnas_modelo].copy()

print(df_modelo.shape)
df_modelo.head()

(96470, 9)


,is_delayed,distancia_km,peso_g,volumen_cm3,customer_state,seller_state,mes,dia_semana,flete
0,0,18.576110,500.0,1976.0,SP,SP,10,0,8.72
1,0,851.495069,400.0,4693.0,BA,SP,7,1,22.76
2,0,514.410666,420.0,9576.0,GO,SP,8,2,19.22
3,0,1822.226336,450.0,6000.0,RN,MG,11,5,27.20
4,0,29.676625,250.0,11475.0,SP,SP,2,1,8.72


## 13. Выводы

- **Фильтрация:** из 99 441 исходного заказа оставлено 96 470 (со статусом `delivered` и известной реальной датой доставки). Отброшено 2 971 заказ (3,0 %).
- **Итоговый датасет:** 96 470 строк (одна строка = один заказ, дубликатов нет) и 9 столбцов: целевая переменная `is_delayed` и 8 признаков.
- **Дисбаланс классов:** с задержкой доставлено 6,77 % заказов (6 534), вовремя — 93,23 % (89 936), то есть примерно 14 к 1. Модель, которая всегда отвечает 0, получила бы accuracy 93,23 %, не найдя ни одной задержки (recall = 0, F1 = 0), поэтому accuracy не подходит. Оцениваем по ROC-AUC, PR-AUC и F1.
- **Пропущенные значения:** среди признаков модели пропуски есть только в `distancia_km` (476 заказов, 0,49 %) из-за почтовых индексов без координат в `geolocation`. В `peso_g` и `volumen_cm3` пропусков нет. Решение: удалить эти 476 строк, так как их меньше 1 %.
- **Выбросы в расстоянии:** максимальное расстояние 8 678 км больше любого возможного расстояния внутри Бразилии (около 4 400 км). Это указывает на ошибочные координаты в `geolocation`. Нужно проверить и отфильтровать координаты вне Бразилии.
- **Утечка данных:** исключены `order_delivered_customer_date` и `dias_retraso`, так как в момент покупки они неизвестны.
- **Следующий шаг:** разделить на train/test со `stratify=y`, закодировать `customer_state` и `seller_state` и обучить baseline — логистическую регрессию (с `class_weight="balanced"` из-за дисбаланса).

### Проверка выбросов:

In [21]:
# сколько заказов с расстоянием больше, чем возможно внутри Бразилии
print("Заказов с расстоянием > 4500 км:", (df["distancia_km"] > 4500).sum())
df["distancia_km"].describe()

Заказов с расстоянием > 4500 км: 4


count    95994.000000
mean       600.788989
std        593.451334
min          0.000000
25%        187.749334
50%        433.878565
75%        797.667754
max       8677.911622
Name: distancia_km, dtype: float64

### Удаление пропусков

In [22]:
df_modelo = df_modelo.dropna(subset=["distancia_km"]).reset_index(drop=True)
print(df_modelo.shape)  # ожидаем (95994, 9)

(95994, 9)
